<a href="https://colab.research.google.com/github/CalculatedContent/xgboost2ww/blob/main/notebooks/XGBWW_Random10_LongRun_Alpha_Tracking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# XGBWW Random-10 Long-Run Alpha Tracking (debug-friendly rewrite)

This notebook is split into small cells so you can debug each phase independently:
1. choose/verify 10 datasets and persist that list;
2. train one model per dataset with exhaustive logging per round;
3. compute WeightWatcher metrics (W1/W2/W7/W8) and train/test accuracy;
4. plot train/test accuracy and alphas vs round.


In [ ]:

# 1) Runtime setup and persistent output folders (Drive-first for Colab resume safety)
from pathlib import Path
import os
import sys

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    drive = None
    IN_COLAB = False

DRIVE_MOUNT_ROOT = Path('/content/drive')
DEFAULT_DRIVE_RUN_ROOT = DRIVE_MOUNT_ROOT / 'MyDrive' / 'xgbww_random10_longrun_alpha_tracking'
LOCAL_FALLBACK_ROOT = Path.cwd() / 'random10_longrun_alpha_tracking'

if IN_COLAB:
    drive.mount(str(DRIVE_MOUNT_ROOT), force_remount=False)
    OUTPUT_ROOT = DEFAULT_DRIVE_RUN_ROOT
else:
    OUTPUT_ROOT = LOCAL_FALLBACK_ROOT

RUNTIME_ROOT = Path.cwd()
REGISTRY_DIR = OUTPUT_ROOT / 'registry'
RUN_DIR = OUTPUT_ROOT / 'run'
PLOTS_DIR = OUTPUT_ROOT / 'plots'
LOGS_DIR = OUTPUT_ROOT / 'logs'
SUMMARIES_DIR = OUTPUT_ROOT / 'summaries'

for p in [OUTPUT_ROOT, REGISTRY_DIR, RUN_DIR, PLOTS_DIR, LOGS_DIR, SUMMARIES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Python:', sys.version.split()[0])
print('Working directory:', RUNTIME_ROOT)
print('Output root:', OUTPUT_ROOT.resolve())
print('In Colab:', IN_COLAB)


In [ ]:
# Colab/bootstrap installs
INCLUDE_KEEL_DATASETS = False  # Optional dataset source; default off for smoother Colab installs

%pip install -q --upgrade pip
%pip install -q xgboost2ww xgbwwdata xgboost weightwatcher torch scikit-learn pandas matplotlib seaborn scipy feather-format pyarrow openml pmlb requests
if INCLUDE_KEEL_DATASETS:
    %pip install -q keel-ds
else:
    print('[INFO] INCLUDE_KEEL_DATASETS=False -> KEEL datasets will be excluded from the registry scan.')

import importlib
import sys

importlib.invalidate_caches()
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('xgboost2ww') or mod_name.startswith('xgbwwdata'):
        sys.modules.pop(mod_name, None)

print('Packages installed and import caches refreshed.')


In [ ]:
# 3) Imports
import json
import time
import traceback
import inspect

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import weightwatcher as ww

from scipy import sparse
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from xgboost2ww import convert
from xgbwwdata import Filters, load_dataset

try:
    from xgbwwdata import scan_datasets as xgbww_scan_datasets
except Exception:
    xgbww_scan_datasets = None

try:
    from xgbwwdata import list_datasets as xgbww_list_datasets
except Exception:
    xgbww_list_datasets = None

print('Imports loaded.')


In [ ]:

# 4) Global config
RANDOM_STATE = 42
DATASET_SAMPLE_SIZE = 10
ADDITIONAL_RANDOM_MODELS = 10
TARGET_MODEL_COUNT = DATASET_SAMPLE_SIZE + ADDITIONAL_RANDOM_MODELS
FAST_SCAN_MODE = True  # optimize registry build for small random samples
FAST_SCAN_SOURCES = ['openml', 'pmlb']  # lighter/faster default source set
ROUND_STRIDE = 25
TOTAL_STEPS = 100
TOTAL_ROUNDS = ROUND_STRIDE * TOTAL_STEPS
SAVE_N_STEPS = 20
N_STEPS = TOTAL_STEPS
TEST_SIZE = 0.2

W_LIST = ['W1', 'W2', 'W7', 'W8']
SELECTED_DATASET_UIDS = []  # optionally pin exact dataset_uids
REUSE_EXISTING_SAMPLE = True
INCLUDE_KEEL_DATASETS = False
MAX_DENSE_ELEMENTS = int(2e8)
RESUME_FROM_CHECKPOINT = True
FORCE_RESTART_ALL = False
FORCE_REBUILD_MANIFEST = False
FORCE_RESTART_RUN = False

# If scanning fails in this runtime, optionally bypass scanning and sample from known-good non-KEEL UIDs.
ALLOW_UID_FALLBACK_IF_SCAN_FAILS = True
KNOWN_GOOD_NON_KEEL_UIDS = [
    'openml:31',
    'openml:37',
    'openml:44',
    'openml:50',
    'openml:54',
    'openml:1461',
    'openml:1485',
    'openml:4534',
    'openml:1590',
    'pmlb:adult',
    'pmlb:credit-g',
    'pmlb:spambase',
    'pmlb:diabetes',
]

# lowercase aliases for compatibility with older cells/snippets
registry_dir = REGISTRY_DIR

SAMPLED_REGISTRY_CSV = REGISTRY_DIR / 'random10_registry.csv'
SAMPLED_REGISTRY_FEATHER = REGISTRY_DIR / 'random10_registry.feather'
RUN_ROOT = RUN_DIR / 'random10_longrun_alpha_tracking_v1'
MANIFEST_PATH = RUN_ROOT / 'manifest.json'
METRICS_CSV = RUN_ROOT / 'metrics.csv'
WW_METRICS_CSV = RUN_ROOT / 'weightwatcher_metrics.csv'
SUMMARY_CSV = RUN_ROOT / 'experiment_summary.csv'

np.random.seed(RANDOM_STATE)
print(f'TOTAL_STEPS={TOTAL_STEPS}, TOTAL_ROUNDS={TOTAL_ROUNDS}, ROUND_STRIDE={ROUND_STRIDE}, SAVE_N_STEPS={SAVE_N_STEPS}, TARGET_MODEL_COUNT={TARGET_MODEL_COUNT}')


In [ ]:

# 5) xgbwwdata scan / registry build + fixed 20-model manifest

import inspect

VALID_SCAN_SOURCES = {'openml', 'pmlb', 'keel', 'libsvm', 'amlb'}


def _materialize_registry(reg):
    if isinstance(reg, pd.DataFrame):
        return reg
    if hasattr(reg, 'to_pandas'):
        return reg.to_pandas()
    return pd.DataFrame(list(reg))


def _call_xgbww_scan(api_func, filters: Filters, sources):
    sig = inspect.signature(api_func)
    kwargs = {}
    if 'filters' in sig.parameters:
        kwargs['filters'] = filters
    if 'sources' in sig.parameters:
        kwargs['sources'] = sources
    if 'smoke_train' in sig.parameters:
        kwargs['smoke_train'] = True
    if 'random_state' in sig.parameters:
        kwargs['random_state'] = RANDOM_STATE
    if 'log_every' in sig.parameters:
        kwargs['log_every'] = 25
    return api_func(**kwargs) if kwargs else api_func()


def build_registry_df(filters: Filters, sources) -> pd.DataFrame:
    if not isinstance(sources, (list, tuple)) or len(sources) == 0:
        raise RuntimeError('scan_datasets must be called with an explicit, non-empty sources list.')

    bad = [s for s in sources if s not in VALID_SCAN_SOURCES]
    if bad:
        raise RuntimeError(f'Invalid scan source names: {bad}. Valid sources are {sorted(VALID_SCAN_SOURCES)}')

    if xgbww_scan_datasets is not None:
        return _materialize_registry(_call_xgbww_scan(xgbww_scan_datasets, filters, list(sources)))
    if xgbww_list_datasets is not None:
        return _materialize_registry(_call_xgbww_scan(xgbww_list_datasets, filters, list(sources)))
    try:
        from xgbwwdata import get_registry
        return _materialize_registry(_call_xgbww_scan(get_registry, filters, list(sources)))
    except Exception as e:
        raise RuntimeError('No compatible xgbwwdata scan/list API found in this environment.') from e


def fallback_registry_from_known_uids(known_uids, sample_size):
    rows = []
    for uid in known_uids:
        if not any(uid.startswith(f'{src}:') for src in ['openml', 'pmlb', 'libsvm', 'amlb']):
            continue
        src = uid.split(':', 1)[0]
        rows.append({'dataset_uid': uid, 'source': src, 'task_type': 'classification'})
    if not rows:
        raise RuntimeError('Known-good UID fallback list is empty after filtering.')
    df = pd.DataFrame(rows)
    return df.sample(n=min(sample_size, len(df)), random_state=RANDOM_STATE).reset_index(drop=True)


filters = Filters(
    min_rows=200,
    max_rows=20000 if FAST_SCAN_MODE else 60000,
    max_features=2000 if FAST_SCAN_MODE else 50000,
    max_dense_elements=min(MAX_DENSE_ELEMENTS, int(5e7)) if FAST_SCAN_MODE else MAX_DENSE_ELEMENTS,
    preprocess=True,
)

SCAN_SOURCES = list(FAST_SCAN_SOURCES) if FAST_SCAN_MODE else ['openml', 'pmlb', 'amlb', 'libsvm']
if INCLUDE_KEEL_DATASETS:
    SCAN_SOURCES.append('keel')

print('Scanning sources:', SCAN_SOURCES, '(FAST_SCAN_MODE=' + str(FAST_SCAN_MODE) + ')')
full_registry_csv = registry_dir / 'full_registry.csv'

if RESUME_FROM_CHECKPOINT and full_registry_csv.exists() and not FORCE_RESTART_ALL:
    full_registry_df = pd.read_csv(full_registry_csv)
else:
    try:
        full_registry_df = build_registry_df(filters, SCAN_SOURCES)
    except Exception as scan_err:
        msg = str(scan_err).lower()
        if ALLOW_UID_FALLBACK_IF_SCAN_FAILS and ('scan' in msg or 'registry' in msg or 'source' in msg or 'api' in msg):
            print('[WARN] Registry scan failed; using known-good UID fallback:', scan_err)
            full_registry_df = fallback_registry_from_known_uids(KNOWN_GOOD_NON_KEEL_UIDS, TARGET_MODEL_COUNT)
        else:
            raise
    full_registry_df.to_csv(full_registry_csv, index=False)

if 'source' in full_registry_df.columns and not INCLUDE_KEEL_DATASETS:
    full_registry_df = full_registry_df[full_registry_df['source'].astype(str).str.lower() != 'keel'].reset_index(drop=True)

registry_df = full_registry_df.copy()
if 'task_type' in registry_df.columns:
    registry_df = registry_df[registry_df['task_type'].astype(str).str.contains('class', case=False, na=False)]
elif 'task' in registry_df.columns:
    registry_df = registry_df[registry_df['task'].astype(str).str.contains('class', case=False, na=False)]

if 'n_classes' in registry_df.columns:
    registry_df = registry_df[(registry_df['n_classes'].fillna(0) >= 2) & (registry_df['n_classes'].fillna(0) <= 20)]

registry_df = registry_df.reset_index(drop=True)
if registry_df.empty:
    raise RuntimeError('Filtered registry is empty; cannot define fixed model manifest.')

RUN_ROOT.mkdir(parents=True, exist_ok=True)


def _atomic_json_write(payload, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = out_path.with_suffix(out_path.suffix + '.tmp')
    with open(tmp_path, 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp_path, out_path)


if MANIFEST_PATH.exists() and not FORCE_REBUILD_MANIFEST:
    manifest = json.loads(MANIFEST_PATH.read_text())
    fixed_models = manifest['models']
    print(f"[RESUME] Loaded manifest for {len(fixed_models)} fixed models")
else:
    if SELECTED_DATASET_UIDS:
        sampled_registry = registry_df[registry_df['dataset_uid'].isin(SELECTED_DATASET_UIDS)].copy()
    elif REUSE_EXISTING_SAMPLE and SAMPLED_REGISTRY_CSV.exists() and not FORCE_REBUILD_MANIFEST:
        sampled_registry = pd.read_csv(SAMPLED_REGISTRY_CSV)
    else:
        sampled_registry = registry_df.sample(n=min(TARGET_MODEL_COUNT, len(registry_df)), random_state=RANDOM_STATE).copy()

    sampled_registry = sampled_registry.drop_duplicates(subset=['dataset_uid']).reset_index(drop=True)
    if len(sampled_registry) < TARGET_MODEL_COUNT:
        raise RuntimeError(f'Need {TARGET_MODEL_COUNT} models but only found {len(sampled_registry)} unique dataset_uids.')

    sampled_registry = sampled_registry.head(TARGET_MODEL_COUNT).reset_index(drop=True)
    sampled_registry.to_csv(SAMPLED_REGISTRY_CSV, index=False)
    try:
        sampled_registry.to_feather(SAMPLED_REGISTRY_FEATHER)
    except Exception as e:
        print('[WARN] Could not save feather registry:', e)

    fixed_models = []
    for i, row in sampled_registry.iterrows():
        fixed_models.append({
            'model_id': int(i + 1),
            'model_dir': f"model_{i+1:03d}",
            'dataset_uid': str(row.get('dataset_uid')),
            'source': str(row.get('source', 'unknown')),
            'random_seed': int(RANDOM_STATE),
            'hyperparameters': {
                'booster': 'gbtree',
                'eta': 0.05,
                'max_depth': 6,
                'subsample': 0.9,
                'colsample_bytree': 0.9,
                'tree_method': 'hist',
            },
            'feature_config': {'scaler': 'StandardScaler', 'with_mean_sparse': False},
            'split_config': {'test_size': float(TEST_SIZE), 'split_seed': int(RANDOM_STATE)},
        })

    manifest = {
        'created_at_utc': pd.Timestamp.utcnow().isoformat(),
        'total_steps': int(TOTAL_STEPS),
        'round_stride': int(ROUND_STRIDE),
        'saveN': int(SAVE_N_STEPS),
        'checkpoint_root_path': str(RUN_ROOT),
        'models': fixed_models,
    }
    _atomic_json_write(manifest, MANIFEST_PATH)
    print(f"[START] Created manifest for {len(fixed_models)} fixed models at {MANIFEST_PATH}")

sampled_registry = pd.DataFrame(fixed_models)
print('Selected model count:', len(sampled_registry))
display(sampled_registry[['model_id', 'dataset_uid', 'source']])


In [ ]:
# 6) Helpers for preprocessing, WW extraction, and structured logging

def detect_task_type(y):
    y_np = np.asarray(y)
    unique = np.unique(y_np)
    n_classes = len(unique)
    if n_classes < 2:
        return 'invalid', n_classes, y_np
    task = 'binary' if n_classes == 2 else 'multiclass'
    return task, n_classes, y_np


def preprocess_split(X, y):
    strat = y if len(np.unique(y)) > 1 else None
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=strat
    )
    scaler = StandardScaler(with_mean=False) if sparse.issparse(X_train) else StandardScaler()
    X_train_s = scaler.fit_transform(X_train).astype(np.float32)
    X_test_s = scaler.transform(X_test).astype(np.float32)
    return X_train_s, X_test_s, np.asarray(y_train), np.asarray(y_test)


def build_params(task, n_classes):
    params = {
        'booster': 'gbtree',
        'eta': 0.05,
        'max_depth': 6,
        'subsample': 0.9,
        'colsample_bytree': 0.9,
        'tree_method': 'hist',
        'seed': RANDOM_STATE,
    }
    if task == 'binary':
        params.update({'objective': 'binary:logistic', 'eval_metric': 'logloss'})
    else:
        params.update({'objective': 'multi:softprob', 'eval_metric': 'mlogloss', 'num_class': int(n_classes)})
    return params


def prediction_to_accuracy(bst, dmat, y_true, task):
    y_prob = bst.predict(dmat)
    if task == 'binary':
        y_pred = (y_prob >= 0.5).astype(np.int32)
    else:
        y_pred = np.argmax(y_prob, axis=1).astype(np.int32)
    return float(accuracy_score(y_true, y_pred))


def build_layer_for_W(bst, W_name, current_round, X_train_for_convert, y_train_np, params):
    multiclass_mode = 'avg' if int(params.get('num_class', 2)) > 2 else 'error'
    if 'torch' not in globals() or torch is None:
        raise RuntimeError("torch is required when return_type='torch'.")
    return convert(
        model=bst,
        data=X_train_for_convert,
        labels=y_train_np,
        W=W_name,
        return_type='torch',
        nfolds=5,
        t_points=min(current_round, 160),
        random_state=RANDOM_STATE,
        train_params=params,
        num_boost_round=current_round,
        multiclass=multiclass_mode,
    )


def ww_alphas_for_all_Ws(model, X_train, y_train, params, boost_round):
    out = {}
    for W in W_LIST:
        key = f'alpha_{W}'
        try:
            converted = build_layer_for_W(model, W, boost_round, X_train, y_train, params)
            details = ww.WeightWatcher(model=converted).analyze(randomize=True, detX=True)
            alpha_val = float(pd.to_numeric(details.get('alpha', pd.Series(dtype=float)), errors='coerce').dropna().mean())
            out[key] = alpha_val
        except Exception:
            out[key] = np.nan
    return out


def append_error(err_rows, dataset_uid, stage, exc):
    err_rows.append({
        'dataset_uid': str(dataset_uid),
        'stage': stage,
        'error_type': type(exc).__name__,
        'error_message': str(exc),
        'traceback': traceback.format_exc(),
        'timestamp_utc': pd.Timestamp.utcnow().isoformat(),
    })


In [ ]:

# 7) Resume-safe execution loop with atomic per-model checkpoints

import shutil

MODELS_DIR = RUN_ROOT / 'models'
LOG_DIR = RUN_ROOT / 'logs'
for p in [RUN_ROOT, MODELS_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)


def _atomic_write_text(text: str, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = out_path.with_suffix(out_path.suffix + '.tmp')
    with open(tmp_path, 'w', encoding='utf-8') as f:
        f.write(text)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp_path, out_path)


def _atomic_write_json(payload, out_path: Path):
    _atomic_write_text(json.dumps(payload, indent=2), out_path)


def _model_tmp_path(path: Path) -> Path:
    return path.with_name(f"{path.stem}.tmp{path.suffix}")


def _load_booster(path: Path):
    bst = xgb.Booster()
    try:
        bst.load_model(str(path))
        return bst
    except xgb.core.XGBoostError as e:
        if path.suffix == '.json':
            legacy_tmp = path.with_suffix('.legacy.ubj')
            shutil.copy2(path, legacy_tmp)
            try:
                bst.load_model(str(legacy_tmp))
                print(f"[RESUME] loaded legacy UBJSON checkpoint from {path.name}")
                return bst
            finally:
                legacy_tmp.unlink(missing_ok=True)
        raise e


def _safe_csv_load(path: Path, required_cols=None):
    if not path.exists():
        return pd.DataFrame(columns=required_cols or [])
    try:
        df = pd.read_csv(path)
    except Exception:
        return pd.DataFrame(columns=required_cols or [])
    if required_cols:
        for c in required_cols:
            if c not in df.columns:
                df[c] = np.nan
        df = df[required_cols]
    return df


def _flush_drive(path: Path):
    try:
        fd = os.open(str(path.parent), os.O_RDONLY)
        try:
            os.fsync(fd)
        finally:
            os.close(fd)
    except Exception:
        pass


def _uid_for_filename(dataset_uid: str) -> str:
    return ''.join(ch if ch.isalnum() or ch in {'-', '_', '.'} else '_' for ch in str(dataset_uid))


def _model_paths(model_id: int):
    model_root = MODELS_DIR / f'model_{model_id:03d}'
    ckpt_dir = model_root / 'checkpoints'
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    return {
        'root': model_root,
        'ckpt': ckpt_dir,
        'state': model_root / 'state.json',
        'final_model': model_root / 'final_model.ubj',
    }


def _checkpoint_files(model_id: int, round_end: int):
    p = _model_paths(model_id)
    return {
        'booster': p['ckpt'] / f'booster_round_{round_end:04d}.ubj',
        'metrics': p['ckpt'] / f'metrics_round_{round_end:04d}.json',
        'ww': p['ckpt'] / f'ww_round_{round_end:04d}.json',
        'state_round': p['ckpt'] / f'state_round_{round_end:04d}.json',
    }


def _checkpoint_complete(model_id: int, round_end: int):
    files = _checkpoint_files(model_id, round_end)
    return all(f.exists() for f in files.values())


def _list_complete_rounds(model_id: int):
    ckpt_dir = _model_paths(model_id)['ckpt']
    rounds = []
    for file in ckpt_dir.glob('state_round_*.json'):
        try:
            round_end = int(file.stem.split('_')[-1])
        except Exception:
            continue
        if _checkpoint_complete(model_id, round_end):
            rounds.append(round_end)
    return sorted(rounds)


def _load_latest_state(model_id: int):
    rounds = _list_complete_rounds(model_id)
    if not rounds:
        return None
    last_round = rounds[-1]
    state_path = _checkpoint_files(model_id, last_round)['state_round']
    return json.loads(state_path.read_text())



def _ww_row_has_required_alphas(row: pd.Series, w_list):
    for w in w_list:
        col = f'alpha_{w}'
        if col not in row.index:
            return False
        if pd.isna(pd.to_numeric(row[col], errors='coerce')):
            return False
    return True

def _append_unique_csv(csv_path: Path, incoming_df: pd.DataFrame, dedupe_cols):
    existing = _safe_csv_load(csv_path)
    merged = pd.concat([existing, incoming_df], ignore_index=True)
    merged = merged.drop_duplicates(subset=dedupe_cols, keep='last').sort_values(dedupe_cols).reset_index(drop=True)
    _atomic_write_text(merged.to_csv(index=False), csv_path)
    _flush_drive(csv_path)


def _save_checkpoint_atomic(model_spec, bst, step, round_end, metrics_payload, ww_payload):
    model_id = int(model_spec['model_id'])
    paths = _checkpoint_files(model_id, round_end)

    booster_tmp = _model_tmp_path(paths['booster'])
    bst.save_model(str(booster_tmp))
    os.replace(booster_tmp, paths['booster'])

    _atomic_write_json(metrics_payload, paths['metrics'])
    _atomic_write_json(ww_payload, paths['ww'])

    state_payload = {
        'model_id': model_id,
        'dataset_uid': model_spec['dataset_uid'],
        'random_seed': int(model_spec['random_seed']),
        'hyperparameters': model_spec['hyperparameters'],
        'feature_config': model_spec['feature_config'],
        'split_config': model_spec['split_config'],
        'current_step': int(step),
        'current_round': int(round_end),
        'last_checkpoint_round': int(round_end),
        'total_steps': int(TOTAL_STEPS),
        'model_file_path': str(paths['booster']),
        'checkpoint_timestamp': pd.Timestamp.utcnow().isoformat(),
    }
    _atomic_write_json(state_payload, paths['state_round'])
    _atomic_write_json(state_payload, _model_paths(model_id)['state'])

    _append_unique_csv(METRICS_CSV, pd.DataFrame([metrics_payload]), dedupe_cols=['model_id', 'round'])
    _append_unique_csv(WW_METRICS_CSV, pd.DataFrame([ww_payload]), dedupe_cols=['model_id', 'round'])
    print(f"[SAVE] metrics written through round {round_end}")


print(f"[RESUME] Loaded manifest for {len(manifest['models'])} fixed models")
metrics_df = _safe_csv_load(METRICS_CSV)
ww_df = _safe_csv_load(WW_METRICS_CSV)
summary_rows = []

for model_spec in manifest['models']:
    model_id = int(model_spec['model_id'])
    dataset_uid = model_spec['dataset_uid']
    source = model_spec.get('source', 'unknown')

    latest_state = _load_latest_state(model_id)
    last_complete_round = int(latest_state['current_round']) if latest_state else 0
    last_complete_step = int(latest_state['current_step']) if latest_state else 0

    if last_complete_step >= TOTAL_STEPS:
        print(f"[SKIP] model_{model_id:02d} already complete through step {TOTAL_STEPS}")
        continue

    try:
        X, y, meta = load_dataset(dataset_uid, filters=filters)
        task, n_classes, y_np = detect_task_type(y)
        if task == 'invalid':
            raise ValueError(f'Invalid target for dataset_uid={dataset_uid}')

        X_train, X_test, y_train, y_test = preprocess_split(X, y_np)
        dtrain, dtest = xgb.DMatrix(X_train, label=y_train), xgb.DMatrix(X_test, label=y_test)
        params = build_params(task, n_classes)

        start_step = last_complete_step + 1
        bst = None
        if last_complete_round > 0:
            booster_path = _checkpoint_files(model_id, last_complete_round)['booster']
            bst = _load_booster(booster_path)
            print(f"[RESUME] model_{model_id:02d} loading {booster_path.name}")
            print(f"[RESUME] model_{model_id:02d} continuing from round {last_complete_round + ROUND_STRIDE}")
        else:
            print(f"[START] model_{model_id:02d} starting at step 1")

        for step in range(start_step, TOTAL_STEPS + 1):
            round_end = step * ROUND_STRIDE

            if _checkpoint_complete(model_id, round_end):
                continue

            bst = xgb.train(params=params, dtrain=dtrain, num_boost_round=ROUND_STRIDE, xgb_model=bst, verbose_eval=False)
            train_acc = prediction_to_accuracy(bst, dtrain, y_train, task)
            test_acc = prediction_to_accuracy(bst, dtest, y_test, task)

            ww_existing = ww_df[(ww_df.get('model_id') == model_id) & (ww_df.get('round') == round_end)] if not ww_df.empty else pd.DataFrame()
            if ww_existing.empty:
                ww_metrics = ww_alphas_for_all_Ws(bst, X_train, y_train, params, round_end)
            else:
                candidate = ww_existing.iloc[0]
                if _ww_row_has_required_alphas(candidate, W_LIST):
                    ww_metrics = candidate.to_dict()
                    print(f"[RESUME] WeightWatcher results already exist through round {round_end}")
                else:
                    print(f"[RESUME] Recomputing WeightWatcher metrics for round {round_end} to match current W_LIST={W_LIST}")
                    ww_metrics = ww_alphas_for_all_Ws(bst, X_train, y_train, params, round_end)

            metrics_payload = {
                'model_id': model_id,
                'dataset_uid': str(dataset_uid),
                'source': str(source),
                'step': int(step),
                'round': int(round_end),
                'train_acc': float(train_acc),
                'test_acc': float(test_acc),
                'alpha_W7': float(ww_metrics.get('alpha_W7', np.nan)),
                'timestamp': pd.Timestamp.utcnow().isoformat(),
            }
            ww_payload = {
                'model_id': model_id,
                'dataset_uid': str(dataset_uid),
                'step': int(step),
                'round': int(round_end),
                **{f'alpha_{w}': float(ww_metrics.get(f'alpha_{w}', np.nan)) for w in W_LIST},
                'timestamp': pd.Timestamp.utcnow().isoformat(),
            }

            should_save = (step % SAVE_N_STEPS == 0) or (step == TOTAL_STEPS)
            if should_save:
                _save_checkpoint_atomic(model_spec, bst, step, round_end, metrics_payload, ww_payload)
                metrics_df = _safe_csv_load(METRICS_CSV)
                ww_df = _safe_csv_load(WW_METRICS_CSV)

        final_round = TOTAL_STEPS * ROUND_STRIDE
        final_model_path = _model_paths(model_id)['final_model']
        final_tmp = _model_tmp_path(final_model_path)
        bst.save_model(str(final_tmp))
        os.replace(final_tmp, final_model_path)
        _flush_drive(final_model_path)

        model_rows = metrics_df[metrics_df['model_id'] == model_id] if not metrics_df.empty else pd.DataFrame()
        final_test_acc = float(model_rows.sort_values('round')['test_acc'].iloc[-1]) if not model_rows.empty else np.nan
        best_test_acc = float(model_rows['test_acc'].max()) if not model_rows.empty else np.nan
        best_round = int(model_rows.loc[model_rows['test_acc'].idxmax(), 'round']) if not model_rows.empty else 0
        final_alpha_w7 = float(model_rows.sort_values('round')['alpha_W7'].iloc[-1]) if not model_rows.empty else np.nan

        summary_rows.append({
            'model_id': model_id,
            'dataset_uid': dataset_uid,
            'final_round': final_round,
            'final_test_acc': final_test_acc,
            'final_alpha_W7': final_alpha_w7,
            'best_test_acc': best_test_acc,
            'best_round': best_round,
        })
        print(f"[DONE] model_{model_id:02d} completed all {TOTAL_STEPS} steps")

    except Exception as e:
        print(f"[ERROR] model_{model_id:02d} {dataset_uid}: {type(e).__name__}: {e}")
        traceback.print_exc()

if summary_rows:
    summary_df = pd.DataFrame(summary_rows)
    _append_unique_csv(SUMMARY_CSV, summary_df, dedupe_cols=['model_id'])

print('[DONE] All 20 models completed 100 steps')
print(f'[DONE] Total rounds per model = {TOTAL_ROUNDS}')
print('[DONE] Checkpoints written successfully')


In [ ]:
# 8) Plot per-dataset metrics after training (debug in a separate cell)
if 'training_df' not in globals() or training_df.empty:
    if TRAINING_LOG_CSV.exists():
        training_df = pd.read_csv(TRAINING_LOG_CSV)
        print('Loaded training_df from disk:', TRAINING_LOG_CSV)
    else:
        raise RuntimeError('No training dataframe available. Run the training cell first.')

for dataset_uid, g in training_df.groupby('dataset_uid'):
    g = g.sort_values('round').reset_index(drop=True)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    axes[0].plot(g['round'], g['train_accuracy'], marker='o', label='train_accuracy')
    axes[0].plot(g['round'], g['test_accuracy'], marker='o', label='test_accuracy')
    axes[0].set_title(f'Accuracy vs Round | {dataset_uid}')
    axes[0].set_xlabel('Round')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    for W in W_LIST:
        col = f'alpha_{W}'
        if col in g.columns:
            axes[1].plot(g['round'], g[col], marker='.', label=col)
    axes[1].set_title(f'WW Alpha vs Round | {dataset_uid}')
    axes[1].set_xlabel('Round')
    axes[1].set_ylabel('Alpha')
    axes[1].legend(ncol=2, fontsize=8)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    png_path = PLOTS_DIR / f'{dataset_uid}_accuracy_alpha_vs_round.png'
    plt.savefig(png_path, dpi=140)
    plt.show()
    plt.close(fig)

    print('Saved plot:', png_path)


In [ ]:
# 9) Optional quick aggregate checks
if 'training_df' not in globals() or training_df.empty:
    training_df = pd.read_csv(TRAINING_LOG_CSV)

best_rows = training_df.loc[training_df.groupby('dataset_uid')['test_accuracy'].idxmax()].sort_values('test_accuracy', ascending=False)
display(best_rows[['dataset_uid', 'round', 'train_accuracy', 'test_accuracy'] + [f'alpha_{w}' for w in W_LIST if f'alpha_{w}' in best_rows.columns]])

summary = {
    'n_datasets_requested': DATASET_SAMPLE_SIZE,
    'n_datasets_trained': int(training_df['dataset_uid'].nunique()) if not training_df.empty else 0,
    'n_rows': int(len(training_df)),
    'training_csv': str(TRAINING_LOG_CSV),
    'registry_csv': str(SAMPLED_REGISTRY_CSV),
    'errors_csv': str(ERROR_LOG_CSV) if ERROR_LOG_CSV.exists() else None,
}
print(json.dumps(summary, indent=2))
